# TGA Data Analysis Example
## Thermogravimetric Analysis of Fractionated Lignin

This notebook demonstrates the TGA/DSC processing pipeline using ChemEng-Toolkit.

In [ ]:
import sys
sys.path.insert(0, "..")
from chemeng_toolkit.thermal_analysis import TGAProcessor

## 1. Load TGA Data

In [ ]:
folder_map = [
    ("../data/tga_sample/gt_100k.xls", ">100k"),
    ("../data/tga_sample/100k-20k.xls", "100k-20k"),
    ("../data/tga_sample/20k-10k.xls", "20k-10k"),
    ("../data/tga_sample/10k-5k.xls", "10k-5k"),
]
processor = TGAProcessor()
data = {}
for path, label in folder_map:
    d = processor.load_from_xls(path)
    if d:
        data[label] = d
processor.data = data
print(f"Loaded {len(data)} samples")

## 2. TG and DTG Curves

In [ ]:
processor.plot_tg(show=True)

In [ ]:
processor.plot_dtg(show=True)

## 3. Feature Parameter Summary

In [ ]:
summary = processor.summary_table(print_table=True)

## 4. Decomposition Stage Analysis

In [ ]:
d = data[">100k"]
stages, summary = processor.find_decomposition_stages(d['temp'], d['weight'], d['dtg'])
print(f"T5%: {summary['T_5%']:.1f} C")
print(f"Residue: {summary['residue']:.2f}%")
for s in stages:
    print(f"  {s['name']}: {s['t_start']:.0f}-{s['t_end']:.0f} C, mass loss = {s['mass_loss']:.2f}%")

## 5. Dual-Axis TGA-DTG Plot

In [ ]:
processor.plot_dual_axis(">100k", show=True)

## 6. Kinetic Analysis

In [ ]:
from chemeng_toolkit.thermal_analysis import run_coats_redfern_batch
best, f1 = run_coats_redfern_batch(data)

## 7. Segmented Kinetics

In [ ]:
from chemeng_toolkit.thermal_analysis import segmented_coats_redfern, plot_ea_vs_alpha
seg_results = {}
for label in data:
    d = data[label]
    seg_results[label] = segmented_coats_redfern(d['temp'], d['weight'])
ea_data = {label: [r['E_kJ'] for r in results if r['E_kJ'] is not None] for label, results in seg_results.items()}
plot_ea_vs_alpha(ea_data, show=True)